In [117]:
import pandas as pd
import numpy as np

# 先讀進模擬資料
data = pd.read_csv(r"C:\Users\USER\Desktop\碩論\程式碼\embedding_data.csv")

data

,X2,X3,X1,Y
0,255.000000,94.940954,162.097863,4.686122
1,215.309400,130.434396,131.702366,4.410815
2,188.692556,112.414072,176.294826,3.444338
3,197.089163,155.840962,81.755694,3.378867
4,208.651847,70.102607,70.191431,3.135581
...,...,...,...,...
95,47.862764,55.539969,181.891681,-2.831205
96,56.421777,89.129206,191.903929,-3.037451
97,60.563777,24.069923,0.000000,-3.492096
98,86.754581,59.616503,135.187503,-3.672475


In [118]:
cols = ["X2","X3","X1"] # 針對變數標準化，後面做softmax的時候，數值才不會爆掉

data[cols] = (data[cols] - data[cols].mean()) / data[cols].std()

data

,X2,X3,X1,Y
0,2.985660,-0.455945,0.902386,4.686122
1,2.183840,0.287470,0.274274,4.410815
2,1.646134,-0.089969,1.195760,3.444338
3,1.815760,0.819613,-0.757854,3.378867
4,2.049346,-0.976188,-0.996825,3.135581
...,...,...,...,...
95,-1.198872,-1.281204,1.311417,-2.831205
96,-1.025966,-0.577673,1.518317,-3.037451
97,-0.942290,-1.940348,-2.447304,-3.492096
98,-0.413190,-1.195821,0.346293,-3.672475


In [119]:
# 為了測試模型架構，先使用第一筆資料
test = data.iloc[0,:3]
y = data.iloc[0,3]

In [120]:
y = np.array(y) # 為了轉tensor，所以serious要轉成array

In [121]:
import torch

shape = (1,8) # 投影成8維，為了方便後續的attention計算

random = torch.randn(shape)

In [122]:
random # 先隨便猜

tensor([[-0.2335, -0.4931, -0.7374,  0.2007,  0.7090,  0.0389, -1.3366, -0.7490]])

In [123]:
test = torch.tensor(test, dtype = torch.float32) # tensor毛好多，統一設定dtype = float32
test = test.reshape(1,3)
test

C:\Users\USER\AppData\Local\Temp\ipykernel_29024\492316973.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  test = torch.tensor(test, dtype = torch.float32) # tensor毛好多，統一設定dtype = float32


tensor([[ 2.9857, -0.4559,  0.9024]])

In [124]:
y = torch.tensor(y, dtype = torch.float32)
y

tensor(4.6861)

In [125]:
random = random.reshape(-1,1) # 為了後續矩陣乘法的需要
random

tensor([[-0.2335],
        [-0.4931],
        [-0.7374],
        [ 0.2007],
        [ 0.7090],
        [ 0.0389],
        [-1.3366],
        [-0.7490]])

In [126]:
E =  random @ test # 將隨便猜的八維向量，根據原始的embedding縮放
E # 每個col是一個某個X，經過映射之後的新embedding

tensor([[-0.6971,  0.1065, -0.2107],
        [-1.4721,  0.2248, -0.4449],
        [-2.2017,  0.3362, -0.6654],
        [ 0.5992, -0.0915,  0.1811],
        [ 2.1169, -0.3233,  0.6398],
        [ 0.1160, -0.0177,  0.0351],
        [-3.9908,  0.6094, -1.2062],
        [-2.2363,  0.3415, -0.6759]])

In [127]:
wq_shape = (4,8) # KQ space先設定4維，反正比embedding的小
wq = torch.randn(wq_shape)
wq

tensor([[ 0.0168, -0.3646,  0.2963,  0.1831, -0.1667, -0.7933, -0.2012,  0.1236],
        [-1.6374,  0.6367, -0.4158,  1.0703,  0.1477,  0.9950, -0.5211, -0.4420],
        [ 0.8605,  1.3054, -2.6652, -1.2994,  2.0624, -0.1905, -0.4856, -1.3577],
        [-0.3321,  1.5369,  1.8507,  0.0448, -1.4670, -1.9341, -0.0524,  0.5495]])

In [128]:
Q = wq @ E
Q

tensor([[ 6.4009e-02, -9.7749e-03,  1.9346e-02],
        [ 5.2569e+00, -8.0279e-01,  1.5888e+00],
        [ 1.1886e+01, -1.8151e+00,  3.5924e+00],
        [-1.0429e+01,  1.5926e+00, -3.1519e+00]])

In [129]:
wk_shape = (4,8)
wk = torch.randn(wq_shape)
wk

tensor([[ 0.5937, -1.4777, -1.3679, -0.3240, -0.5242,  1.1176, -0.6345, -0.0146],
        [-0.3837,  0.4518,  0.2407,  0.3344,  0.1898, -1.5249,  1.8068, -0.2206],
        [-0.8175,  0.8086, -0.0751, -0.6294,  0.8537,  0.1092,  1.3518, -1.5280],
        [ 0.3634, -1.3185, -0.7832,  1.9632, -0.5886,  0.0518,  0.1485, -0.6604]])

In [130]:
K = wk @ E
K

tensor([[ 6.1636, -0.9413,  1.8629],
        [-7.2192,  1.1025, -2.1819],
        [-0.9899,  0.1512, -0.2992],
        [ 4.2326, -0.6464,  1.2793]])

In [131]:
print(Q)
print(K)

tensor([[ 6.4009e-02, -9.7749e-03,  1.9346e-02],
        [ 5.2569e+00, -8.0279e-01,  1.5888e+00],
        [ 1.1886e+01, -1.8151e+00,  3.5924e+00],
        [-1.0429e+01,  1.5926e+00, -3.1519e+00]])
tensor([[ 6.1636, -0.9413,  1.8629],
        [-7.2192,  1.1025, -2.1819],
        [-0.9899,  0.1512, -0.2992],
        [ 4.2326, -0.6464,  1.2793]])


In [135]:
aa = K[:,1] * Q[:,2]
aa.sum() 

tensor(4.3138)

In [150]:
scores = K.T @ Q 

dk = 4 # QK space的維度

scores = scores/np.sqrt(dk)
scores

tensor([[-46.7309,   7.1364, -14.1239],
        [  7.1364,  -1.0898,   2.1569],
        [-14.1239,   2.1569,  -4.2688]])

In [152]:
import torch.nn.functional as F

attn = F.softmax(scores, dim=0)

attn # row是K、col是Q

tensor([[4.0343e-24, 9.9291e-01, 8.4845e-08],
        [1.0000e+00, 2.6567e-04, 9.9838e-01],
        [5.8449e-10, 6.8290e-03, 1.6168e-03]])

In [160]:
E

tensor([[-0.6971,  0.1065, -0.2107],
        [-1.4721,  0.2248, -0.4449],
        [-2.2017,  0.3362, -0.6654],
        [ 0.5992, -0.0915,  0.1811],
        [ 2.1169, -0.3233,  0.6398],
        [ 0.1160, -0.0177,  0.0351],
        [-3.9908,  0.6094, -1.2062],
        [-2.2363,  0.3415, -0.6759]])

In [159]:
wv_shape = (8,8)
wv = torch.randn(wv_shape)
wv

tensor([[ 0.8302,  0.2216, -2.1865, -1.0313, -1.0464,  0.4707,  1.0368,  0.9642],
        [-1.1850, -0.4377, -0.3945, -0.2708, -0.9413,  0.4967,  0.3190,  0.6616],
        [ 0.3660,  0.3601,  0.6358, -1.3027, -0.3305, -0.6579,  1.7718, -1.5455],
        [ 0.0048, -1.1722, -1.2465,  0.8133,  0.0712, -0.5870, -1.5549,  1.3753],
        [ 1.2415, -1.3926,  0.5680,  0.4234, -0.1279,  0.3269, -0.3149,  1.2073],
        [ 0.1842,  0.5996, -0.5460, -1.7087,  0.3750,  0.8656,  0.0161, -2.0327],
        [-1.7830, -0.3766, -0.7292,  0.4901,  0.3167, -1.7698, -1.5047,  0.7545],
        [ 0.5794,  1.0510, -1.4818,  0.5609, -1.8410, -0.8691, -0.4297, -0.4279]])

In [166]:
wv[0,:] @ E[:,0]

tensor(-5.1636)

In [165]:
V = wv @ E
V

tensor([[-5.1636,  0.7885, -1.5606],
        [-2.5110,  0.3835, -0.7589],
        [-7.3561,  1.1234, -2.2233],
        [ 8.1665, -1.2471,  2.4683],
        [-1.4881,  0.2273, -0.4498],
        [ 4.5427, -0.6937,  1.3730],
        [ 8.4793, -1.2949,  2.5628],
        [ 0.3214, -0.0491,  0.0971]])

In [167]:
attn

tensor([[4.0343e-24, 9.9291e-01, 8.4845e-08],
        [1.0000e+00, 2.6567e-04, 9.9838e-01],
        [5.8449e-10, 6.8290e-03, 1.6168e-03]])

In [182]:
V[0,:] @ attn[:,0]

tensor(0.7885)

In [178]:
delta_E = V @ attn
delta_E

tensor([[ 0.7885, -5.1374,  0.7847],
        [ 0.3835, -2.4983,  0.3816],
        [ 1.1234, -7.3188,  1.1180],
        [-1.2471,  8.1251, -1.2411],
        [ 0.2273, -1.4806,  0.2262],
        [-0.6937,  4.5197, -0.6904],
        [-1.2949,  8.4363, -1.2886],
        [-0.0491,  0.3198, -0.0488]])

In [179]:
E

tensor([[-0.6971,  0.1065, -0.2107],
        [-1.4721,  0.2248, -0.4449],
        [-2.2017,  0.3362, -0.6654],
        [ 0.5992, -0.0915,  0.1811],
        [ 2.1169, -0.3233,  0.6398],
        [ 0.1160, -0.0177,  0.0351],
        [-3.9908,  0.6094, -1.2062],
        [-2.2363,  0.3415, -0.6759]])

In [ ]:
New_E = E + delta_E
New_E # 新的embedding

tensor([[ 0.0914, -5.0309,  0.5740],
        [-1.0887, -2.2735, -0.0633],
        [-1.0784, -6.9826,  0.4525],
        [-0.6479,  8.0336, -1.0600],
        [ 2.3442, -1.8038,  0.8660],
        [-0.5777,  4.5020, -0.6553],
        [-5.2857,  9.0457, -2.4948],
        [-2.2854,  0.6613, -0.7248]])